# 金融 CoT/PoT 可验证强化学习：SFT2 -> Fino1/FinCoT + Core GRPO

本 notebook 从已经训练好的 `sft2_dual_merged` 出发，探索一条 Fino1-style CoT supplement 与 FinQA/ConvFinQA PoT reward 结合的强化学习路线。

核心判断：CoT / PoT 是推理表示形式，distill-R1/o1 是推理能力迁移方法，RL/GRPO 是推理能力激励方法。当前项目不重新训练 SFT1/SFT2，也不使用本地 distill 数据；它保留 `sft2_dual_merged` 的 program-supervised 能力，引入公开 `TheFinAI/Fino1_Reasoning_Path_FinQA` 与 `TheFinAI/FinCoT` 作为低比例 CoT supplement，再用 Program/answer 可验证 reward 做 GRPO。

目标：
- 使用 `/root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged` 作为 policy 初始化。
- 保持 FinQA/ConvFinQA core program-verifiable 数据为主干，不让外部 CoT 覆盖 gold `Program`。
- 将 Fino1 Reasoning Path FinQA 和 FinCoT 统一为 `cot_answer_only` 样本，允许 `Program: N/A`。
- 对 core 样本优化答案正确性、程序一致性、格式稳定性、证据 grounding 与简洁推理。
- 对外部 CoT 样本只使用 answer/format/brevity/negative-response avoidance，不施加 program reward。

当前 DPO 不作为 RL 起点，因为此前 pass@k 对比中 `dpo_passk` 没有超过 `sft2_merged_passk`.


## 框架选择

默认框架：**TRL `GRPOTrainer`**。

TRL 适合当前阶段的原因是它能在单机 LoRA/PEFT 场景下快速验证数据混合与 Python reward 是否有效。verl 和 OpenRLHF 更适合后续大规模 rollout、vLLM/SGLang 推理和 remote reward server，但第一阶段不应该先迁移框架，而应该先确认 reward contract 是否真正提升 FinQA/ConvFinQA benchmark。

本 notebook 与 `docs/fin_pot_cot_rl.md` 对齐：先保留 `sft2_dual_merged` 作为 program-supervised baseline，再将 Fino1/FinCoT 作为 CoT supplement，最后通过 program-verifiable GRPO 把 pass@k 中已有的正确 Program 推向更高概率输出。


In [ ]:
import json
import math
import os
import random
import re
import subprocess
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda devices:', torch.cuda.device_count())
except Exception as exc:
    print('torch import failed:', repr(exc))

try:
    import datasets
    print('datasets:', datasets.__version__)
except Exception as exc:
    print('datasets import failed:', repr(exc))

try:
    import trl
    print('trl:', trl.__version__)
except Exception as exc:
    print('trl import failed. Install/update TRL before GRPO training:', repr(exc))


In [ ]:
BASE_MODEL = Path('/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct')
SFT2_MERGED_OUT = Path('/root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged')

RL_DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning_v2/rl')
FINO1_CACHE_DIR = RL_DATA_ROOT / 'fino1_finqa_path'
FINCOT_CACHE_DIR = RL_DATA_ROOT / 'fincot'
GRPO_TRAIN_FILE = RL_DATA_ROOT / 'train_cot_pot_grpo_mixed.jsonl'
GRPO_VALID_FILE = RL_DATA_ROOT / 'valid_cot_pot_grpo_mixed.jsonl'
GRPO_SMOKE_FILE = RL_DATA_ROOT / 'smoke_cot_pot_grpo_mixed.jsonl'

GRPO_OUT = Path('/root/autodl-tmp/outputs/financial_reasoning_v2/grpo_cot_pot_lora')
GRPO_BENCH_OUT = Path('/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/grpo_cot_pot_passk')

FINQA_TRAIN_STRICT = Path('/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft1_dual_strict.jsonl')
SFT2_CONV_STRICT = Path('/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_convfinqa_turn_dual_strict.jsonl')
SFT2_FINQA_REPLAY = Path('/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_finqa_replay_dual.jsonl')

FINQA_EVAL_FILE = Path('/root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json')
CONVFINQA_EVAL_FILE = Path('/root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json')
DESIGN_DOC = Path('/root/MedicalGPT/docs/fin_pot_cot_rl.md')

for path in [RL_DATA_ROOT, FINO1_CACHE_DIR, FINCOT_CACHE_DIR, GRPO_OUT, GRPO_BENCH_OUT]:
    path.mkdir(parents=True, exist_ok=True)

print('BASE_MODEL exists:', BASE_MODEL.exists(), BASE_MODEL)
print('SFT2_MERGED_OUT exists:', SFT2_MERGED_OUT.exists(), SFT2_MERGED_OUT)
print('DESIGN_DOC exists:', DESIGN_DOC.exists(), DESIGN_DOC)
print('RL_DATA_ROOT:', RL_DATA_ROOT)


## 数据集方案

本 notebook 的混合 GRPO 数据集包含三类来源。

第一类是 **核心 program-verifiable 数据**，来自本地 FinQA / ConvFinQA v2 strict 文件，保留当前 MedicalGPT 的优势，即 gold `Program` 与 `Normalized Answer`。这部分样本使用 `program_numeric` reward profile，是 Program execution RL 的主干。

第二类是 **Fino1 Reasoning Path FinQA**，来自 `TheFinAI/Fino1_Reasoning_Path_FinQA`。它更贴近 FinQA 风格，用于提供 FinQA-style CoT supplement。除非后续完成本地 FinQA 对齐，否则它不强行补 `Program`，统一写 `Program: N/A`，使用 `cot_answer_only` reward profile。

第三类是 **FinCoT RL split**，来自 `TheFinAI/FinCoT`。它提供更广金融 reasoning diversity 和 negative response 字段，用 answer/format/brevity/negative-response avoidance 作为轻量 reward，同样不参与 program reward。

本 notebook 明确忽略本地 distill 数据，避免把旧 `<think>/<answer>` 产物和新的 Fino1/FinCoT 公开数据路线混在一起。


In [ ]:
SEED = 42
MAX_CORE_RL_ROWS = 3500
MAX_FINO1_FINQA_ROWS = 750
MAX_FINCOT_RL_ROWS = 750
VALID_RATIO = 0.05
CORE_CONV_TO_FINQA_RATIO = 2.0

FINO1_FINQA_PATH_DATASET = 'TheFinAI/Fino1_Reasoning_Path_FinQA'
FINO1_FINQA_SPLIT = 'train'
FINCOT_DATASET = 'TheFinAI/FinCoT'
FINCOT_SPLIT = 'RL'
USE_FINO1_FINQA = True
USE_FINCOT = True

random.seed(SEED)
print({
    'MAX_CORE_RL_ROWS': MAX_CORE_RL_ROWS,
    'MAX_FINO1_FINQA_ROWS': MAX_FINO1_FINQA_ROWS,
    'MAX_FINCOT_RL_ROWS': MAX_FINCOT_RL_ROWS,
    'VALID_RATIO': VALID_RATIO,
    'CORE_CONV_TO_FINQA_RATIO': CORE_CONV_TO_FINQA_RATIO,
    'USE_FINO1_FINQA': USE_FINO1_FINQA,
    'USE_FINCOT': USE_FINCOT,
})


In [ ]:
def read_jsonl(path: Path, max_rows: Optional[int] = None) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        print('missing:', path)
        return rows
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows is not None and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            count += 1
    return count


def sample_rows(rows: List[Dict[str, Any]], n: Optional[int], seed: int = SEED) -> List[Dict[str, Any]]:
    rows = list(rows)
    rng = random.Random(seed)
    rng.shuffle(rows)
    if n is None or n < 0 or n >= len(rows):
        return rows
    return rows[:n]


def split_train_valid(rows: List[Dict[str, Any]], valid_ratio: float, seed: int = SEED) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    rows = sample_rows(rows, None, seed)
    valid_n = max(1, int(round(len(rows) * valid_ratio))) if rows else 0
    return rows[valid_n:], rows[:valid_n]


def first_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, str):
        return value
    return str(value)


In [ ]:
ANCHORS = ['Evidence:', 'Reasoning:', 'Program:', 'Answer:', 'Normalized Answer:']
PROGRAM_OPS = ['add', 'subtract', 'multiply', 'divide', 'greater', 'table_max', 'table_min', 'table_sum', 'table_average', 'average', 'sum', 'max', 'min']
NUMBER_RE = re.compile(r'-?\d+(?:,\d{3})*(?:\.\d+)?%?')


def extract_anchor(text: str, anchor: str) -> str:
    if not text or anchor not in text:
        return ''
    start = text.index(anchor) + len(anchor)
    rest = text[start:]
    next_positions = [rest.find(a) for a in ANCHORS if a != anchor and rest.find(a) >= 0]
    end = min(next_positions) if next_positions else len(rest)
    return rest[:end].strip().split('\n')[0].strip()


def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        if 'content' in completion:
            return first_text(completion.get('content'))
        if 'text' in completion:
            return first_text(completion.get('text'))
    if isinstance(completion, list) and completion:
        return completion_text(completion[0])
    return first_text(completion)


def normalize_number(text: str) -> Optional[float]:
    text = first_text(text)
    if not text:
        return None
    matches = NUMBER_RE.findall(text.replace(',', ''))
    if not matches:
        return None
    raw = matches[-1]
    is_percent = raw.endswith('%')
    if is_percent:
        raw = raw[:-1]
    try:
        value = float(raw)
    except ValueError:
        return None
    return value / 100.0 if is_percent else value


def numeric_equal(pred: str, gold: str, abs_tol: float = 1e-4, rel_tol: float = 1e-4) -> bool:
    pred_num = normalize_number(pred)
    gold_num = normalize_number(gold)
    if pred_num is None or gold_num is None:
        return first_text(pred).strip().lower() == first_text(gold).strip().lower()
    return abs(pred_num - gold_num) <= max(abs_tol, abs(gold_num) * rel_tol)


def program_ops(program: str) -> List[str]:
    program = first_text(program).lower()
    return [op for op in PROGRAM_OPS if re.search(rf'\b{re.escape(op)}\s*\(', program)]


In [ ]:
def sharegpt_to_grpo(row: Dict[str, Any], source_dataset: str) -> Optional[Dict[str, Any]]:
    conv = row.get('conversations') or []
    if len(conv) < 2:
        return None
    prompt = conv[0].get('value') or conv[0].get('content') or ''
    target = conv[1].get('value') or conv[1].get('content') or ''
    if not prompt or not target:
        return None
    meta = row.get('metadata') or {}
    answer = meta.get('answer_norm') or extract_anchor(target, 'Normalized Answer:') or meta.get('answer_exe') or extract_anchor(target, 'Answer:')
    gold_program = meta.get('program_canonical') or meta.get('program_raw') or extract_anchor(target, 'Program:')
    if answer is None or first_text(answer) == '':
        return None
    return {
        'prompt': prompt,
        'answer': first_text(answer),
        'gold_program': first_text(gold_program),
        'reference_response': target,
        'source_dataset': source_dataset,
        'task_type': 'program_verifiable',
        'reward_profile': 'program_numeric',
        'record_id': first_text(meta.get('record_id') or row.get('id') or meta.get('raw_id')),
        'metadata': {'source_path': source_dataset},
    }


finqa_candidates = []
for path in [SFT2_FINQA_REPLAY, FINQA_TRAIN_STRICT]:
    finqa_candidates.extend([x for x in (sharegpt_to_grpo(r, 'finqa') for r in read_jsonl(path)) if x])

conv_candidates = [x for x in (sharegpt_to_grpo(r, 'convfinqa_turn') for r in read_jsonl(SFT2_CONV_STRICT)) if x]

# Deduplicate by prompt to avoid overweighting replay duplicates.
def dedupe_by_prompt(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for row in rows:
        key = row['prompt']
        if key in seen:
            continue
        seen.add(key)
        out.append(row)
    return out

finqa_candidates = dedupe_by_prompt(finqa_candidates)
conv_candidates = dedupe_by_prompt(conv_candidates)

conv_target = int(round(MAX_CORE_RL_ROWS * CORE_CONV_TO_FINQA_RATIO / (CORE_CONV_TO_FINQA_RATIO + 1)))
finqa_target = MAX_CORE_RL_ROWS - conv_target
core_rows = sample_rows(conv_candidates, conv_target, SEED + 1) + sample_rows(finqa_candidates, finqa_target, SEED + 2)
core_rows = sample_rows(core_rows, None, SEED + 3)

print('FinQA candidates:', len(finqa_candidates), 'target:', finqa_target)
print('ConvFinQA candidates:', len(conv_candidates), 'target:', conv_target)
print('Core rows:', len(core_rows))
pd.DataFrame(core_rows[:3])


In [ ]:
def build_external_prompt(question: str) -> str:
    return (
        question.strip()
        + '\n\nOutput format:\n'
        + 'Evidence:\n- Not provided.\n\n'
        + 'Reasoning:\n...\n\n'
        + 'Program: N/A\n\n'
        + 'Answer: ...\n'
        + 'Normalized Answer: ...\n'
    )


def load_fino1_finqa_path(max_rows: int = MAX_FINO1_FINQA_ROWS) -> List[Dict[str, Any]]:
    if not USE_FINO1_FINQA:
        return []
    try:
        from datasets import load_dataset
        ds = load_dataset(FINO1_FINQA_PATH_DATASET, split=FINO1_FINQA_SPLIT, cache_dir=str(FINO1_CACHE_DIR))
    except Exception as exc:
        print('Could not load Fino1 Reasoning Path FinQA. Rerun with network access or set USE_FINO1_FINQA=False.')
        print(repr(exc))
        return []

    rows = []
    for rec in ds:
        question = first_text(rec.get('Open-ended Verifiable Question') or rec.get('Question') or rec.get('question'))
        answer = first_text(rec.get('Ground-True Answer') or rec.get('Ground-Truth Answer') or rec.get('Answer') or rec.get('answer'))
        reasoning = first_text(rec.get('Complex_CoT') or rec.get('Complex CoT') or rec.get('Reasoning_process') or rec.get('reasoning'))
        response = first_text(rec.get('Response') or rec.get('Final_response') or rec.get('response'))
        if not question or not answer:
            continue
        rows.append({
            'prompt': build_external_prompt(question),
            'answer': answer,
            'gold_program': '',
            'reference_reasoning': reasoning,
            'reference_response': response,
            'negative_reasoning': '',
            'negative_response': '',
            'source_dataset': 'fino1_finqa_path',
            'task_type': 'cot_answer_only',
            'reward_profile': 'cot_answer_only',
            'record_id': first_text(rec.get('id') or rec.get('question_id')),
            'metadata': {'program_available': False},
        })
    return sample_rows(rows, max_rows, SEED + 4)


def load_fincot_rl(max_rows: int = MAX_FINCOT_RL_ROWS) -> List[Dict[str, Any]]:
    if not USE_FINCOT:
        return []
    try:
        from datasets import load_dataset
        ds = load_dataset(FINCOT_DATASET, split=FINCOT_SPLIT, cache_dir=str(FINCOT_CACHE_DIR))
    except Exception as exc:
        print('Could not load FinCoT RL split. Rerun with network access or set USE_FINCOT=False.')
        print(repr(exc))
        return []

    rows = []
    for rec in ds:
        question = first_text(rec.get('Question'))
        answer = first_text(rec.get('Answer') or rec.get('Final_response'))
        if not question or not answer:
            continue
        rows.append({
            'prompt': build_external_prompt(question),
            'answer': answer,
            'gold_program': '',
            'reference_reasoning': first_text(rec.get('Reasoning_process')),
            'reference_response': first_text(rec.get('Final_response')),
            'negative_reasoning': first_text(rec.get('Negative_reasoning_process')),
            'negative_response': first_text(rec.get('Negative_response')),
            'source_dataset': 'fincot_rl',
            'task_type': 'cot_answer_only',
            'reward_profile': 'cot_answer_only',
            'record_id': '',
            'metadata': {'program_available': False},
        })
    return sample_rows(rows, max_rows, SEED + 5)


fino1_rows = load_fino1_finqa_path()
fincot_rows = load_fincot_rl()
print('Fino1 Reasoning Path FinQA rows:', len(fino1_rows))
print('FinCoT RL rows:', len(fincot_rows))
pd.DataFrame((fino1_rows + fincot_rows)[:5])


In [ ]:
print('External CoT sources:')
print('Fino1 dataset:', FINO1_FINQA_PATH_DATASET, 'split:', FINO1_FINQA_SPLIT, 'enabled:', USE_FINO1_FINQA)
print('FinCoT dataset:', FINCOT_DATASET, 'split:', FINCOT_SPLIT, 'enabled:', USE_FINCOT)
print('Local distill data is intentionally ignored in this notebook.')
print('Design doc:', DESIGN_DOC)

source_contract = pd.DataFrame([
    {'source': 'finqa/convfinqa_turn', 'reward_profile': 'program_numeric', 'program_reward': True, 'role': 'core PoT / verifiable RL'},
    {'source': 'fino1_finqa_path', 'reward_profile': 'cot_answer_only', 'program_reward': False, 'role': 'FinQA-style CoT supplement'},
    {'source': 'fincot_rl', 'reward_profile': 'cot_answer_only', 'program_reward': False, 'role': 'broad financial CoT supplement'},
])
display(source_contract)


In [ ]:
mixed_rows = sample_rows(core_rows + fino1_rows + fincot_rows, None, SEED + 6)
train_rows, valid_rows = split_train_valid(mixed_rows, VALID_RATIO, SEED + 7)
smoke_rows = sample_rows(mixed_rows, 20, SEED + 8)

write_jsonl(GRPO_TRAIN_FILE, train_rows)
write_jsonl(GRPO_VALID_FILE, valid_rows)
write_jsonl(GRPO_SMOKE_FILE, smoke_rows)

summary = pd.DataFrame(mixed_rows).groupby(['source_dataset', 'reward_profile']).size().reset_index(name='rows') if mixed_rows else pd.DataFrame()
print('train:', len(train_rows), GRPO_TRAIN_FILE)
print('valid:', len(valid_rows), GRPO_VALID_FILE)
print('smoke:', len(smoke_rows), GRPO_SMOKE_FILE)
display(summary)
display(pd.DataFrame(smoke_rows[:5])[['source_dataset', 'reward_profile', 'answer', 'gold_program']])


## Reward 函数

Reward 配置：

| 配置 | 适用数据 | 主要 reward |
|---|---|---|
| `program_numeric` | FinQA / ConvFinQA | 标准化答案、程序算子、程序-答案一致性、格式、证据、简洁性 |
| `cot_answer_only` | Fino1 / FinCoT | 答案、Reasoning/格式、简洁推理、避免复述负样本回答 |

Fino1/FinCoT 样本没有 gold program，因此不因为 `Program: N/A` 被惩罚。正式 GRPO 前，应将 core 样本的轻量 program consistency reward 升级为严格 DSL execution reward。


In [ ]:
def reward_answer(completions, answer, reward_profile=None, **kwargs):
    rewards = []
    for completion, gold, profile in zip(completions, answer, reward_profile or ['program_numeric'] * len(completions)):
        text = completion_text(completion)
        pred = extract_anchor(text, 'Normalized Answer:') or extract_anchor(text, 'Answer:') or text
        weight = 0.50 if profile == 'program_numeric' else 0.60
        rewards.append(weight if numeric_equal(pred, gold) else 0.0)
    return rewards


def reward_format(completions, reward_profile=None, **kwargs):
    rewards = []
    for completion, profile in zip(completions, reward_profile or ['program_numeric'] * len(completions)):
        text = completion_text(completion)
        required = ['Evidence:', 'Reasoning:', 'Answer:']
        if profile == 'program_numeric':
            required.extend(['Program:', 'Normalized Answer:'])
        else:
            required.append('Program:')
        score = sum(1 for anchor in required if anchor in text) / len(required)
        weight = 0.10 if profile == 'program_numeric' else 0.15
        rewards.append(score * weight)
    return rewards


def reward_program(completions, gold_program=None, reward_profile=None, **kwargs):
    rewards = []
    gold_program = gold_program or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, gold, profile in zip(completions, gold_program, reward_profile):
        if profile != 'program_numeric':
            rewards.append(0.0)
            continue
        text = completion_text(completion)
        pred_program = extract_anchor(text, 'Program:')
        gold_ops = program_ops(gold)
        pred_ops = program_ops(pred_program)
        if not gold_ops or not pred_program or pred_program.strip().upper() == 'N/A':
            rewards.append(0.0)
            continue
        op_score = sum(1 for op in set(gold_ops) if op in set(pred_ops)) / len(set(gold_ops))
        rewards.append(op_score * 0.20)
    return rewards


def reward_program_answer_consistency(completions, answer=None, reward_profile=None, **kwargs):
    # TODO: replace this lightweight check with strict DSL execution before full-scale RL.
    rewards = []
    answer = answer or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, gold, profile in zip(completions, answer, reward_profile):
        if profile != 'program_numeric':
            rewards.append(0.0)
            continue
        text = completion_text(completion)
        program = extract_anchor(text, 'Program:')
        pred = extract_anchor(text, 'Normalized Answer:') or extract_anchor(text, 'Answer:')
        if program and program.strip().upper() != 'N/A' and normalize_number(pred) is not None:
            rewards.append(0.10 if numeric_equal(pred, gold) else 0.03)
        else:
            rewards.append(0.0)
    return rewards


def reward_evidence(completions, prompt=None, reward_profile=None, **kwargs):
    rewards = []
    prompt = prompt or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, pr, profile in zip(completions, prompt, reward_profile):
        if profile != 'program_numeric':
            rewards.append(0.0)
            continue
        evidence = extract_anchor(completion_text(completion), 'Evidence:')
        ev_nums = set(NUMBER_RE.findall(evidence.replace(',', '')))
        prompt_nums = set(NUMBER_RE.findall(first_text(pr).replace(',', '')))
        rewards.append(0.05 if ev_nums and ev_nums.intersection(prompt_nums) else 0.0)
    return rewards


def reward_brevity_and_relevance(completions, negative_response=None, reward_profile=None, **kwargs):
    rewards = []
    negative_response = negative_response or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, neg, profile in zip(completions, negative_response, reward_profile):
        text = completion_text(completion)
        words = text.split()
        if profile == 'program_numeric':
            rewards.append(0.05 if 20 <= len(words) <= 180 else 0.0)
        else:
            score = 0.0
            if 25 <= len(words) <= 220:
                score += 0.15
            neg_text = first_text(neg).strip()
            if neg_text and neg_text[:80] not in text:
                score += 0.10
            elif not neg_text:
                score += 0.05
            rewards.append(score)
    return rewards


REWARD_FUNCS = [
    reward_answer,
    reward_format,
    reward_program,
    reward_program_answer_consistency,
    reward_evidence,
    reward_brevity_and_relevance,
]
print([fn.__name__ for fn in REWARD_FUNCS])


In [ ]:
smoke_completions = [
    'Evidence:\n- the 2007 value is 991.1 and the 2008 value is 959.2\n\nReasoning:\nCompute the year-over-year change.\n\nProgram: divide(subtract(959.2, 991.1), 991.1)\n\nAnswer: -3.2%\nNormalized Answer: -0.03219',
    'Evidence:\n- Not provided.\n\nReasoning:\nUse the provided financial reasoning path and produce the final response.\n\nProgram: N/A\n\nAnswer: 10%\nNormalized Answer: 0.10',
    'This answer has no structure.'
]
smoke_kwargs = {
    'answer': ['-0.03219', '0.10', '0.5'],
    'gold_program': ['divide(subtract(959.2, 991.1), 991.1)', '', ''],
    'reward_profile': ['program_numeric', 'cot_answer_only', 'program_numeric'],
    'prompt': ['2007 991.1 2008 959.2', 'external prompt', 'prompt'],
    'negative_response': ['', 'wrong answer text', ''],
}
for fn in REWARD_FUNCS:
    print(fn.__name__, fn(smoke_completions, **smoke_kwargs))


## GRPO 训练

下面的训练 cell 默认是安全的：`RUN_GRPO_SMOKE` 和 `RUN_FULL_GRPO` 都是 `False`。请先打开 smoke run，确认数据和 reward smoke test 正常后，再启动完整训练。


In [ ]:
GRPO_NUM_GENERATIONS = 4
MAX_PROMPT_LENGTH = 1536
MAX_COMPLETION_LENGTH = 384
LEARNING_RATE = 5e-6
BETA = 0.001
MAX_STEPS = 300
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
USE_VLLM = False

RUN_GRPO_SMOKE = False
RUN_FULL_GRPO = False

print({
    'GRPO_NUM_GENERATIONS': GRPO_NUM_GENERATIONS,
    'MAX_PROMPT_LENGTH': MAX_PROMPT_LENGTH,
    'MAX_COMPLETION_LENGTH': MAX_COMPLETION_LENGTH,
    'LEARNING_RATE': LEARNING_RATE,
    'BETA': BETA,
    'MAX_STEPS': MAX_STEPS,
    'USE_VLLM': USE_VLLM,
})


In [ ]:
def run_trl_grpo(train_file: Path, valid_file: Path, output_dir: Path, max_steps: int):
    from datasets import load_dataset
    from peft import LoraConfig
    from transformers import AutoTokenizer
    from trl import GRPOConfig, GRPOTrainer

    tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL), trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_ds = load_dataset('json', data_files=str(train_file), split='train')
    eval_ds = load_dataset('json', data_files=str(valid_file), split='train') if valid_file.exists() else None

    def to_chat_prompt(example):
        return {
            'prompt': [
                {'role': 'system', 'content': 'You are a financial numerical reasoning assistant. Follow the requested schema exactly.'},
                {'role': 'user', 'content': example['prompt']},
            ]
        }

    train_ds = train_ds.map(to_chat_prompt)
    if eval_ds is not None:
        eval_ds = eval_ds.map(to_chat_prompt)

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    )

    args = GRPOConfig(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        beta=BETA,
        max_steps=max_steps,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_generations=GRPO_NUM_GENERATIONS,
        max_prompt_length=MAX_PROMPT_LENGTH,
        max_completion_length=MAX_COMPLETION_LENGTH,
        logging_steps=10,
        save_steps=100,
        eval_strategy='steps' if eval_ds is not None else 'no',
        eval_steps=50,
        bf16=True,
        report_to='tensorboard',
        remove_unused_columns=False,
        use_vllm=USE_VLLM,
        log_completions=True,
    )

    trainer = GRPOTrainer(
        model=str(SFT2_MERGED_OUT),
        reward_funcs=REWARD_FUNCS,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return trainer


if RUN_GRPO_SMOKE:
    smoke_out = GRPO_OUT.parent / 'grpo_fincot_lora_smoke'
    trainer = run_trl_grpo(GRPO_SMOKE_FILE, GRPO_SMOKE_FILE, smoke_out, max_steps=1)
elif RUN_FULL_GRPO:
    trainer = run_trl_grpo(GRPO_TRAIN_FILE, GRPO_VALID_FILE, GRPO_OUT, max_steps=MAX_STEPS)
else:
    print('Training skipped. Set RUN_GRPO_SMOKE=True first, then RUN_FULL_GRPO=True for the full run.')


## Benchmark 命令

这些命令沿用之前的 pass@k 设置，确保 `grpo_cot_pot_passk` 可以和 `sft2_merged_passk`、`dpo_passk` 直接对比。评估仍以 FinQA/ConvFinQA 为核心，外部 Fino1/FinCoT 只作为训练补充，不作为主 benchmark。


In [ ]:
RUN_BENCHMARK = False

benchmark_cmd = [
    'python', '-m', 'evaluation.evaluate_financial_benchmarks',
    '--tokenizer_path', str(BASE_MODEL),
    '--model_entry', f'sft2={SFT2_MERGED_OUT}',
    '--model_entry', f'grpo={SFT2_MERGED_OUT}',
    '--adapter_entry', f'grpo={GRPO_OUT}',
    '--finqa_test_file', str(FINQA_EVAL_FILE),
    '--convfinqa_test_file', str(CONVFINQA_EVAL_FILE),
    '--finqa_max_samples', '8',
    '--convfinqa_max_samples', '8',
    '--max_new_tokens', '1024',
    '--pass_k', '1,4,8',
    '--num_samples_per_example', '8',
    '--sample_temperature', '0.7',
    '--sample_top_p', '0.95',
    '--sample_seed', '42',
    '--output_dir', str(GRPO_BENCH_OUT),
]

print(' '.join(benchmark_cmd))
if RUN_BENCHMARK:
    subprocess.run(benchmark_cmd, check=True, cwd='/root/MedicalGPT')
else:
    print('Benchmark skipped. Set RUN_BENCHMARK=True after a GRPO adapter exists.')


## 结果分析

这一节在 GRPO 结果可用后，对比已有 run 与新的 GRPO run。


In [ ]:
SUMMARY_FILES = {
    'base_passk': Path('/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/base_passk/benchmark_summary.csv'),
    'sft2_merged_passk': Path('/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft2_merged_passk/benchmark_summary.csv'),
    'dpo_passk': Path('/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/dpo_passk/benchmark_summary.csv'),
    'grpo_cot_pot_passk': GRPO_BENCH_OUT / 'benchmark_summary.csv',
}
frames = []
for run_name, path in SUMMARY_FILES.items():
    if not path.exists():
        print('missing summary:', run_name, path)
        continue
    df = pd.read_csv(path)
    df.insert(0, 'run_name', run_name)
    frames.append(df)
if frames:
    all_summary = pd.concat(frames, ignore_index=True)
    display(all_summary)
else:
    print('No benchmark summaries available yet.')


In [ ]:
def load_greedy_predictions(path: Path) -> Dict[Tuple[str, str], Dict[str, Any]]:
    rows = {}
    if not path.exists():
        return rows
    for row in read_jsonl(path):
        if row.get('generation_mode') == 'greedy':
            rows[(row.get('task_name', ''), row.get('record_id', ''))] = row
    return rows

sft2_preds = load_greedy_predictions(Path('/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft2_merged_passk/sft2_predictions.jsonl'))
grpo_preds = load_greedy_predictions(GRPO_BENCH_OUT / 'grpo_predictions.jsonl')
common_keys = sorted(set(sft2_preds).intersection(grpo_preds))
print('common greedy predictions:', len(common_keys))
for key in common_keys[:5]:
    sft2 = sft2_preds[key]
    grpo = grpo_preds[key]
    print('\n', key)
    print('sft2 correct:', sft2.get('answer_correct'), 'grpo correct:', grpo.get('answer_correct'))
    print('gold:', sft2.get('gold_answer'))
    print('sft2:', first_text(sft2.get('prediction'))[:500])
    print('grpo:', first_text(grpo.get('prediction'))[:500])


## 扩展说明

如果单机 TRL GRPO 相比 `sft2_dual_merged` 有提升，后续应优先扩展同一套数据与 reward 设计，而不是立刻更换研究问题。

后续最重要的两个工程扩展是：第一，将 core 样本的轻量 program consistency reward 替换为严格 DSL execution reward；第二，使用 pass@k mining 构造 hard-but-verifiable GRPO 数据，而不是随机采样普通 SFT 样本。

- **verl**：适合需要 FSDP/vLLM/SGLang placement、自定义 reward manager，以及更细粒度 rollout/training 组件控制的实验。
- **OpenRLHF**：适合需要 Ray + vLLM、远程 reward server、dynamic filtering，以及更大规模 actor/reference/reward 部署的实验。
- 迁移框架时保持 reward contract 不变：`prompt`、`answer`、`gold_program`、`reward_profile`，以及可选 `reference_reasoning`、`negative_response` 字段。

在确认 reward 与混合数据确实提升 benchmark 指标之前，不要急着迁移训练框架。
